# 05 | Classical ML Baselines: HARTH Activity Classification

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Reuse subject-independent partitions from eight HARTH v2.0 participants, compare Dummy, Logistic Regression and Random Forest on **Validation only**. The Test partition remains untouched.

**Defined task:** Eight documented activities (codes 1–8). Cycling categories and undocumented codes are outside this restricted task, not remapped. Class exclusions and source coverage are reported. This is an exploratory baseline, not an independently validated HARTH benchmark.

## 01. Imports and source functions

Python functions are maintained in `.py` files. Short English comments document each Notebook operation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay

# Locate the project and make shared and study-specific modules available.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent: raise RuntimeError("Notebook must run within the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from harth_data import ACTIVITY_NAMES, ensure_harth_sample, load_harth_subject
from core.signal_preprocessing import create_signal_windows
from core.feature_engineering import extract_signal_features
from core.data_splitting import split_group_indices
from core.ml_baselines import (
    train_validation_classifiers, classification_class_report,
    classification_confusion,
)
print("Shared ML utilities imported successfully")

## 02. Prepare data from eight independent HARTH subjects

Identical participants, label-purity criteria and 2-second windows are used as in Notebook 04. Unknown source codes are retained for auditing at loading time, not considered authorised training labels.

In [ ]:
SUBJECTS = ["S023", "S024", "S025", "S027", "S028", "S032", "S034", "S035"]
TASK_CLASSES = tuple(range(1, 9))
DATA_ROOT = root / "data/raw/harth_v2"

# Process each participant independently to preserve boundary information.
feature_parts, metadata_parts = [], []
for subject in SUBJECTS:
    source_path = ensure_harth_sample(DATA_ROOT, subject=subject)
    data, channels = load_harth_subject(
        source_path, allow_unmapped_labels=True
    )
    windows, labels, info = create_signal_windows(
        signals=data[channels].to_numpy(),
        labels=data["label"].to_numpy(),
        window_size=100, step_size=50,
        timestamps=data["timestamp"], sampling_rate=50,
        min_label_purity=1.0, subject_id=subject,
    )
    feature_parts.append(extract_signal_features(windows, channel_names=channels))
    metadata_parts.append(info)

features_all = pd.concat(feature_parts, ignore_index=True)
metadata_all = pd.concat(metadata_parts, ignore_index=True)
assert len(features_all) == len(metadata_all)
print("Complete source feature matrix:", features_all.shape)

## 03. Define the research cohort before partitioning

Only eight predefined documented activities are part of this pilot. Filtering occurs **after time-safe windowing** so removed activities cannot cause windows to bridge source gaps. Exclusions are audited by original class and participant.

In [ ]:
# Audit every source code before the restricted-task filter.
raw_class_counts = (
    metadata_all.groupby(["subject_id", "label"])
    .size().rename("windows").reset_index()
)
raw_class_counts["activity"] = raw_class_counts["label"].map(
    lambda value: ACTIVITY_NAMES.get(int(value), "UNDOCUMENTED")
)
raw_class_counts["in_task"] = raw_class_counts["label"].isin(TASK_CLASSES)
display(raw_class_counts)

# Preserve only the predefined eight activity labels.
included = metadata_all["label"].isin(TASK_CLASSES).to_numpy()
excluded = metadata_all.loc[~included, "label"].value_counts().sort_index()
features = features_all.loc[included].reset_index(drop=True)
metadata = metadata_all.loc[included].reset_index(drop=True)
print("Excluded windows by source code:", excluded.to_dict())
print("Eight-class feature matrix:", features.shape)

## 04. Recreate subject-disjoint Train, Validation and Test

The grouping function is deterministic and uses the same eight subjects and random seed as Notebook 04. **Test rows are reserved but never accessed during model selection or evaluation in this notebook.**

In [ ]:
partitions = split_group_indices(
    metadata["subject_id"].to_numpy(),
    train_fraction=0.50, validation_fraction=0.25,
    test_fraction=0.25, random_state=42,
)

# Check subject membership before extracting supervised arrays.
group_sets = {
    part: set(metadata.iloc[idx]["subject_id"])
    for part, idx in partitions.items()
}
assert not (group_sets["train"] & group_sets["validation"])
assert not (group_sets["train"] & group_sets["test"])
assert not (group_sets["validation"] & group_sets["test"])
display(pd.DataFrame([
    {"partition":part,"subjects":", ".join(sorted(group_sets[part])),
     "windows":len(idx)}
    for part,idx in partitions.items()
]))

# Build ONLY training and validation arrays.
train_ids, validation_ids = partitions["train"], partitions["validation"]
X_train = features.iloc[train_ids]
y_train = metadata.iloc[train_ids]["label"].to_numpy()
X_validation = features.iloc[validation_ids]
y_validation = metadata.iloc[validation_ids]["label"].to_numpy()

# Verify fixed task labels are available for learning and evaluation.
coverage = pd.DataFrame({
    "train": pd.Series(y_train).value_counts(),
    "validation": pd.Series(y_validation).value_counts(),
}).reindex(TASK_CLASSES, fill_value=0).fillna(0).astype(int)
coverage.index = [ACTIVITY_NAMES[c] for c in TASK_CLASSES]
display(coverage)
assert (coverage["train"] > 0).all(), "Not every defined task class is in Train."
print("Held-out test partition: reserved, not inspected for modelling")

## 05. Train and compare baseline classifiers on Validation

`DummyClassifier` provides a necessary majority-class reference. Logistic Regression uses `StandardScaler` fitted **only on Train**; Random Forest uses no scaler. Primary comparison metric is **Macro F1**, which gives each activity equal importance.

In [ ]:
# Fit only on training observations, then evaluate on validation observations.
models, comparison, predictions = train_validation_classifiers(
    X_train, y_train, X_validation, y_validation,
    classes=TASK_CLASSES, random_state=42,
)
display(comparison.style.format({
    name:"{:.3f}" for name in comparison.columns if name != "model"
}))
best_name = comparison.loc[0, "model"]
print("Best validation Macro F1:", best_name)
print("Final Test evaluated: NO")

## 06. Compare validation metrics

All metrics are computed from the **same** validation subjects. These are not final benchmark scores and should not be reported as clinical or external validation.

In [ ]:
# Show model-wise balanced accuracy and macro F1 together.
plot_table = comparison.set_index("model")[["macro_f1", "balanced_accuracy"]]
ax = plot_table.plot(kind="bar", figsize=(9, 4.5), rot=0)
ax.set(title="HARTH: validation-only classifier comparison",
       xlabel="Model", ylabel="Score (0 to 1)", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 07. Inspect the best validation model by activity

A confusion matrix and class-wise F1 show whether aggregate performance hides errors on rare activities. The model selection is based only on Validation.

In [ ]:
# Calculate class-wise metrics with fixed activity names.
best_predictions = predictions[best_name]
class_names = [ACTIVITY_NAMES[code] for code in TASK_CLASSES]
class_report = classification_class_report(
    y_validation, best_predictions,
    classes=TASK_CLASSES, class_names=class_names,
)
display(class_report)

# Display validation confusion matrix in documented class order.
matrix = classification_confusion(
    y_validation, best_predictions, classes=TASK_CLASSES
)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=class_names
).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"HARTH Validation confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

## 08. Scientific quality gate

This notebook is an **exploratory eight-subject, eight-class validation comparison**. Classes outside the predefined task were explicitly excluded. Final Test remains untouched. Hyperparameter selection across many splits, full HARTH participant coverage and uncertainty analysis are future tasks.

In [ ]:
# Verify alignment and leakage protections after model comparison.
assert X_train.shape[1] == X_validation.shape[1] == 30
assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_validation.to_numpy()).all()
assert not (group_sets["train"] & group_sets["validation"])
assert len(comparison) == 3
assert comparison["macro_f1"].between(0, 1).all()

print("Train subject IDs:", sorted(group_sets["train"]))
print("Validation subject IDs:", sorted(group_sets["validation"]))
print("Defined classification task:", [ACTIVITY_NAMES[c] for c in TASK_CLASSES])
print("Undocumented/source-excluded windows:", int((~included).sum()))
print("Final Test model inference performed: NO")
print("Exploratory validation baseline: PASSED")